In [1]:
print("Everything is working fine!")

Everything is working fine!


In [2]:
# import dependencies

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


e:\CAW-GLB\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\anant\AppData\Local\Temp\ipykernel_2820\1395040266.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
# Load documents and split into chunks
loader = PyPDFLoader("E:\\CAW-GLB\\RAG\\PEFT2312.12148v1.pdf")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=75)
docs = text_splitter.split_documents(documents)

In [4]:
print(len(docs))

202


In [5]:
# Embeddings model
embeddings = OllamaEmbeddings(
    model = "nomic-embed-text-v2-moe",
    dimensions=768
)

In [6]:
embeddings

OllamaEmbeddings(model='nomic-embed-text-v2-moe', dimensions=768, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [9]:
vdb = Chroma(
    persist_directory="./chroma_db_Vermaanant",
    embedding_function=embeddings
)

In [10]:
vectorstore = vdb.from_documents(
    documents=docs, 
    embedding=embeddings,
    persist_directory="./chroma_db_Vermaanant",
    collection_name="ollama_test")

In [13]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    seach_kwargs={"k": 5}
)

In [14]:
retriever.invoke("What is the main topic of the paper?")

[Document(metadata={'total_pages': 20, 'title': '', 'page_label': '19', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': 'E:\\CAW-GLB\\RAG\\PEFT2312.12148v1.pdf', 'author': '', 'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'subject': '', 'page': 18, 'creationdate': '2023-12-20T02:07:04+00:00', 'moddate': '2023-12-20T02:07:04+00:00', 'trapped': '/False'}, page_content='[90] D. J. MacKay, “A practical bayesian framework for backpropagation\nnetworks,” Neural Comput., vol. 4, no. 3, pp. 448–472, 1992.\n[91] J. Liu, A. Moreau, M. Preuss, J. Rapin, B. Roziere, F. Teytaud, and\nO. Teytaud, “Versatile black-box optimization,” in Proc. of the 2020\nGenet. and Evolut. Comput. Conf. , 2020, pp. 620–628.\n[92] G. Ilharco, M. T. Ribeiro, M. Wortsman, L. Schmidt, H. Hajishirzi, and\nA. Farhadi, “Editing models with task arithmetic,” in Proc. Int. Conf.\nLearn. Representations, 2023.\n[93] J. Zhan

In [22]:
local_llm =ChatOllama(
    model = 'gemma4:e4b ',
    max_tokens=512,
    temperature=0.3
)

In [23]:
local_llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, model='gemma4:e4b ', temperature=0.3)

In [17]:
prompt_template = """Use the following retrieved documents to answer the question. If you don't know the answer, say I don't know."""


In [18]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(prompt_template)

prompt

PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="Use the following retrieved documents to answer the question. If you don't know the answer, say I don't know.")

In [24]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=local_llm,
    chain_type = "stuff",
    retriever=retriever,
    verbose=True)

In [25]:
qa_chain

RetrievalQA(verbose=True, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the user's question.\nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]), llm=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, model='gemma4:e4b ', temperature=0.3), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_co

In [27]:
qa_chain.invoke("What is LoRA?")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What is LoRA?',
 'result': 'Based on the provided context, LoRA appears to be a technique used for **fine-tuning** large models, particularly in relation to updating pretrained weights.\n\nThe context highlights that LoRA is the foundation for several advanced methods, including:\n\n*   **Parameter-Efficient Fine-tuning:** It is used in methods like "LoRA-guided Pretrained Weight Update" and "LoRA-based Multi-task Fine-tuning," where it helps fine-tune models by updating weights.\n*   **Quantization:** It is the basis for "QLoRA," a quantized variant that addresses computational resource limitations by quantizing the transformer model.\n*   **Adaptability:** Variations like DyLoRA address limitations of the original LoRA by allowing the rank to be adjusted dynamically, rather than being fixed.\n*   **Memory Efficiency:** Techniques like LoRA-FA are proposed to reduce the expensive activation memory associated with LoRA.\n\nIn essence, the context describes LoRA as a core met

In [30]:
results = qa_chain.invoke("What is the main topic of the paper?")

print(results['result'])



> Entering new RetrievalQA chain...

> Finished chain.
The provided text snippets are abstracts and citations from various research papers in the field of artificial intelligence, particularly focusing on natural language processing (NLP). 

It's difficult to pinpoint a single "main topic" across all these excerpts as they cover diverse subfields within NLP.  Some recurring themes include:

* **Model Efficiency:** Papers like [93] and [82] explore techniques to make large language models more efficient by pruning or optimizing their parameters.
* **Cross-lingual Transfer:**  Papers like [109] and [108] investigate methods for adapting pre-trained models to different languages, enabling zero-shot cross-lingual transfer.
* **Black-box Optimization:** [91] delves into techniques for optimizing complex machine learning models without requiring explicit understanding of their internal workings.
* **Model Interpretation:** Papers like [83] and [84] aim to shed light on how pre-trained lang